# APBS electrostatics: protein-ligand complex

Amber-parameterized APBS electrostatic potential map for a docked
protein-ligand complex (`complex.pdb`: protein chain A + ligand chain B,
residue `l01`). This mirrors the first APBS-calculation part of
`examples/browndye/complex_pqr.ipynb`, but parameterizes every body with
AmberTools (no PDB2PQR).

Pipeline:

1. Verify protonation and fix the protein chain (PropKa + PDBFixer).
2. Assign ligand bond orders from a SMILES template and write an SDF.
3. Parameterize the complex with AmberTools/GAFF2 (`pdb4amber`, `obabel`,
   `antechamber`, `parmchk2`, `tleap`, ParmEd PQR export) -> protein, ligand,
   and complex topologies.
4. Solve APBS on the complex PQR to produce `complex.dx`.

Activate the AmberTools environment before launching Jupyter:

```bash
conda activate ambertools
```

In [ ]:
import os
import shutil
from pathlib import Path

import parmed as pmd
from Bio.PDB import PDBIO, PDBParser
from rdkit import Chem

from mdpp.prep import (
    ChainSelect,
    assign_topology,
    fix_pdb,
    run_propka,
    write_apbs_input,
)

# Input (this directory).
EXAMPLE_DIR = Path.cwd()
COMPLEX_PDB = EXAMPLE_DIR / "complex.pdb"

# Workspace layout: each stage owns a folder under tmp/, with transient
# tool files kept in <stage>/intermediate/.
TMP_ROOT = EXAMPLE_DIR / "tmp"
PREP_DIR = TMP_ROOT / "prep"
PREP_INTERMEDIATE = PREP_DIR / "intermediate"
AMBERTOOLS_DIR = TMP_ROOT / "ambertools"
AMBERTOOLS_INTERMEDIATE = AMBERTOOLS_DIR / "intermediate"
APBS_DIR = TMP_ROOT / "apbs"
APBS_INTERMEDIATE = APBS_DIR / "intermediate"
for path in (
    PREP_DIR,
    PREP_INTERMEDIATE,
    AMBERTOOLS_DIR,
    AMBERTOOLS_INTERMEDIATE,
    APBS_DIR,
    APBS_INTERMEDIATE,
):
    path.mkdir(parents=True, exist_ok=True)

# Ligand template (assigns bond orders to the PDB ligand block).
LIGAND_SMILES = (
    r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)"
    r"[C@@H](O)[C@H]3O"
)
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

# AmberTools (complex parameterization).
PROTEIN_FF = "leaprc.protein.ff19SB"
LIGAND_FF = "leaprc.gaff2"
PB_RADII = "mbondi3"
STRIP_PROTEIN_H = True  # let tleap re-add hydrogens per ff19SB

# APBS physics defaults live in mdpp.prep.apbs; override per-call below if a
# system needs non-standard values.

# Export every shell-facing knob so the %%bash cells inherit the same
# configuration without re-typing defaults.
os.environ.update({
    "AMBERTOOLS_DIR": str(AMBERTOOLS_DIR),
    "AMBERTOOLS_INTERMEDIATE": str(AMBERTOOLS_INTERMEDIATE),
    "APBS_DIR": str(APBS_DIR),
    "APBS_INTERMEDIATE": str(APBS_INTERMEDIATE),
    "PROTEIN_FF": PROTEIN_FF,
    "LIGAND_FF": LIGAND_FF,
    "PB_RADII": PB_RADII,
    "STRIP_PROTEIN_H": "1" if STRIP_PROTEIN_H else "0",
    "PH": f"{PH}",
})

## Step 1: Verify protonation and fix protein chain

Extract the protein chain, run PropKa as an explicit pKa/protonation-state
check at the target pH, then add missing residues/atoms/hydrogens via PDBFixer.

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]
chains = {chain.id: chain for chain in model}

pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract protein chain into the prep intermediate folder.
protein_pdb = PREP_INTERMEDIATE / "protein.pdb"
pdb_io.save(str(protein_pdb), ChainSelect(PROTEIN_CHAIN_ID))

# Verify protonation with PropKa before PDBFixer assigns hydrogens.
propka_result = run_propka(protein_pdb)
nonstandard = propka_result.get_nonstandard(PH)
propka_report = PREP_DIR / "protein_propka.tsv"
with propka_report.open("w") as f:
    f.write(
        "residue_type\tres_num\tchain_id\tpka\tmodel_pka\tpropka_protonated\tmodel_protonated\n"
    )
    for residue in propka_result.residues:
        f.write(
            f"{residue.residue_type}\t{residue.res_num}\t{residue.chain_id}\t"
            f"{residue.pka:.3f}\t{residue.model_pka:.3f}\t"
            f"{residue.is_protonated_at(PH)}\t{residue.is_default_protonated_at(PH)}\n"
        )

print(f"PropKa residues checked: {len(propka_result.residues)} -> {propka_report}")
if nonstandard:
    print(f"PropKa predicts {len(nonstandard)} non-standard protonation state(s) at pH {PH}:")
    for residue in nonstandard:
        print(
            f"  {residue.label}: pKa={residue.pka:.2f}, model={residue.model_pka:.2f}, "
            f"PropKa={'protonated' if residue.is_protonated_at(PH) else 'deprotonated'}, "
            f"model={'protonated' if residue.is_default_protonated_at(PH) else 'deprotonated'}"
        )
else:
    print(f"PropKa agrees with model-pKa defaults for all titratable residues at pH {PH}.")

protein_fixed_pdb = PREP_DIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 2: Assign ligand topology

Extract the ligand chain, assign bond orders from the SMILES template, and
write an SDF for antechamber. The SMILES is needed because PDB files lack
bond-order information.

In [ ]:
# Extract ligand chain to the prep intermediate folder for RDKit parsing.
ligand_pdb = PREP_INTERMEDIATE / "ligand.pdb"
pdb_io.save(str(ligand_pdb), ChainSelect(LIGAND_CHAIN_ID))

# Assign bond orders from the SMILES template.
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"Ligand net charge: {ligand_net_charge}")

mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
mol_assigned = assign_topology(mol, template_mol)

# Read the ligand residue name from the docked ligand chain.
lig_resnames = sorted({res.get_resname().strip() for res in chains[LIGAND_CHAIN_ID].get_residues()})
lig_resname = lig_resnames[0]
mol_assigned.SetProp("_Name", lig_resname)

ligand_sdf = PREP_DIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)

(PREP_DIR / "ligand_charge.txt").write_text(f"{ligand_net_charge}\n")
(PREP_DIR / "ligand_resname.txt").write_text(f"{lig_resname}\n")

# Make resname + net charge available to subsequent bash cells.
os.environ["LIG_RESNAME"] = lig_resname
os.environ["NET_CHARGE"] = str(ligand_net_charge)
print(f"Ligand SDF -> {ligand_sdf}")

## Step 3: Parameterize the complex with AmberTools

Run the standard AmberTools pipeline on the prepared protein and ligand:

1. `pdb4amber` cleans the protein PDB for tleap.
2. `obabel` converts the ligand SDF to mol2 as an antechamber seed; we then
   patch the residue name to match the ligand chain.
3. `antechamber` assigns AM1-BCC charges and GAFF2 atom types.
4. `parmchk2` generates any missing GAFF2 parameters.
5. `tleap` combines the protein (ff19SB) and ligand (GAFF2) into a complex
   topology and saves prmtop/rst7 for protein, ligand, and complex bodies.
6. ParmEd exports each topology to PQR (charge + radius per atom).

Outputs live in `tmp/ambertools/`; transient files in its `intermediate/` folder.

In [ ]:
# Stage the Step 1/2 outputs into the AmberTools intermediate folder.
shutil.copy(protein_fixed_pdb, AMBERTOOLS_INTERMEDIATE / "protein_fixed.pdb")
shutil.copy(ligand_sdf, AMBERTOOLS_INTERMEDIATE / "ligand.sdf")

In [ ]:
%%bash
set -euo pipefail
cd "$AMBERTOOLS_INTERMEDIATE"

echo "=== 1. pdb4amber ==="
pdb4amber_args=(-i protein_fixed.pdb -o protein_amber.pdb -d --no-conect)
if [[ "$STRIP_PROTEIN_H" == "1" ]]; then
    pdb4amber_args+=(-y)
fi
pdb4amber "${pdb4amber_args[@]}"

echo "=== 2a. obabel: SDF -> mol2 ==="
obabel ligand.sdf -O ligand_seed.mol2

In [ ]:
# Patch the residue name written by obabel so it matches the docked ligand.
ligand_seed_mol2 = AMBERTOOLS_INTERMEDIATE / "ligand_seed.mol2"
text = ligand_seed_mol2.read_text()
for old in ("UNL1", "UNL", "UNK"):
    text = text.replace(old, lig_resname)
if lig_resname not in text:
    raise RuntimeError(
        f"Residue name {lig_resname!r} not found in {ligand_seed_mol2} after replacement"
    )
ligand_seed_mol2.write_text(text)
print(f"Patched residue name -> {lig_resname}")

In [ ]:
%%bash
set -euo pipefail
cd "$AMBERTOOLS_INTERMEDIATE"

echo "=== 2b. antechamber (AM1-BCC + GAFF2) ==="
antechamber \
    -i ligand_seed.mol2 -fi mol2 \
    -o ligand_amber.mol2 -fo mol2 \
    -c bcc -s 2 -at gaff2 \
    -nc "$NET_CHARGE" -rn "$LIG_RESNAME"

echo "=== 2c. parmchk2 (missing GAFF2 parameters) ==="
parmchk2 -i ligand_amber.mol2 -f mol2 -o ligand.frcmod

echo "=== 3. tleap (ff19SB protein + GAFF2 ligand -> complex) ==="
cat >tleap.in <<EOF
source $PROTEIN_FF
source $LIGAND_FF

$LIG_RESNAME = loadmol2 ligand_amber.mol2
loadamberparams ligand.frcmod
protein = loadpdb protein_amber.pdb
complex = combine {protein $LIG_RESNAME}

set default PBRadii $PB_RADII
saveamberparm protein protein.prmtop protein.rst7
saveamberparm $LIG_RESNAME ligand.prmtop ligand.rst7
saveamberparm complex complex.prmtop complex.rst7
quit
EOF
tleap -f tleap.in

In [ ]:
# ParmEd: convert each topology/coord pair to PQR (charge + radius per atom).
for stem in ("protein", "ligand", "complex"):
    parm = pmd.load_file(
        str(AMBERTOOLS_INTERMEDIATE / f"{stem}.prmtop"),
        xyz=str(AMBERTOOLS_INTERMEDIATE / f"{stem}.rst7"),
    )
    parm.save(str(AMBERTOOLS_INTERMEDIATE / f"{stem}.pqr"), overwrite=True)
    for suffix in ("prmtop", "rst7", "pqr"):
        shutil.copy(
            AMBERTOOLS_INTERMEDIATE / f"{stem}.{suffix}",
            AMBERTOOLS_DIR / f"{stem}.{suffix}",
        )

print("AmberTools PQR outputs:")
for stem in ("protein", "ligand", "complex"):
    print("  ", AMBERTOOLS_DIR / f"{stem}.pqr")

## Step 4: Solve APBS for the complex

Generate an APBS input file (`complex.in`) for the complex PQR and run `apbs`
to produce the electrostatic potential map `complex.dx`. The grid is sized
from the radius-inflated atom bounding box of `complex.pqr` and padded by the
fine/coarse padding defaults in `mdpp.prep.apbs`.

Protein-only and ligand-only diagnostic maps are optional - the dedicated
`protein/` and `ligand/` example notebooks compute them, or uncomment the
lines below to also map the individual bodies here.

In [ ]:
# Stage complex.pqr into the APBS intermediate folder and write complex.in.
shutil.copy(AMBERTOOLS_DIR / "complex.pqr", APBS_INTERMEDIATE / "complex.pqr")
apbs_in = write_apbs_input("complex", APBS_INTERMEDIATE)
print(f"APBS input -> {apbs_in}")

# Optional diagnostics (uncomment to also map protein and ligand alone):
# for diag in ("protein", "ligand"):
#     shutil.copy(AMBERTOOLS_DIR / f"{diag}.pqr", APBS_INTERMEDIATE / f"{diag}.pqr")
#     write_apbs_input(diag, APBS_INTERMEDIATE)

os.environ["STEMS"] = "complex"

In [ ]:
%%bash
set -euo pipefail
cd "$APBS_INTERMEDIATE"

for stem in $STEMS; do
    echo "=== APBS: $stem ==="
    apbs "$stem.in" 2>&1 | tee "$stem.apbs.log"

    if [[ -s "$stem-PE0.dx" ]]; then
        mv "$stem-PE0.dx" "$stem.dx"
    elif [[ -s "$stem.pqr.dx" ]]; then
        mv "$stem.pqr.dx" "$stem.dx"
    fi

    if [[ ! -s "$stem.dx" ]]; then
        echo "Missing $stem.dx" >&2
        exit 1
    fi
    cp "$stem.in" "$APBS_DIR/$stem.in"
    cp "$stem.apbs.log" "$APBS_DIR/$stem.apbs.log"
    cp "$stem.dx" "$APBS_DIR/$stem.dx"
done

ls -lh "$APBS_DIR"/*.dx